In [7]:
import sys
import os

# Add the parent directory (src) to the system path
# The '..' tells it to look one folder up from where the notebook is currently running
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [8]:
# Get the data
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [10]:
#Load the document and build a minisearch index for it
from ingest import  build_index
from evaluation_helper import get_documents


documents = get_documents(filer="llm-zoomcamp")
index = build_index(documents)

ConnectionError: HTTPSConnectionPool(host='datatalks.club', port=443): Max retries exceeded with url: /faq/json/courses.json (Caused by NameResolutionError("HTTPSConnection(host='datatalks.club', port=443): Failed to resolve 'datatalks.club' ([Errno 11001] getaddrinfo failed)"))

In [ ]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [ ]:
# Compute the relevance of the docuemnt
def compute_relevance_text(doc):
    doc_id = doc["document"]
    results = text_search(query=doc["question"])

    relevance = []

    for result in results:
        relevance.append(int(result["id"] == doc_id))

    return relevance

In [ ]:
#check for 1
relevance = compute_relevance_text(ground_truth[0])
relevance

In [ ]:
#Compute for all the records
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

#Change the relevance computation to use a generic search function

def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [ ]:
# Compute the relevance for all the records using a generic search function
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [ ]:
ground_truth_sample = ground_truth[:20]

relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total

In [ ]:
#Compute the relevance for all the ground truth documents
relevance_total = compute_relevance_total(ground_truth, text_search)

### Search Metrics: Hit rate and Mean Reciprocal Rank(MRR)

In [ ]:
example = [
    [1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
]

In [ ]:
#Hit
cnt = 0
for line in example:
    if 1 in line:
        cnt += 1
hit_rate = cnt/len(example)

print(f"Hit rate: {hit_rate:.2%}")

In [ ]:
#MRR
total_score = 0.0
for line in example:
    for rank in range(len(line)):
        if line[rank] == 1:
            total_score += 1/(rank + 1)
            break
        
mrr = total_score / len(example)
print(f"MRR: {mrr:.4f}")

In [ ]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [ ]:
mrr(example)
# 0.822

In [ ]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [ ]:
evaluate(
    ground_truth,
    text_search
)